IMPORTS

In [4]:
import os
import sys
import tarfile
import zipfile
from collections import defaultdict
from io import StringIO
from PIL import Image
import numpy as np
import six.moves.urllib as urllib
import matplotlib.pyplot as plt
import imageio
import tensorflow as tf


# Ensure research and object_detection directories are in sys.path
research_dir = os.path.abspath(os.path.join('..', 'models', 'research'))
obj_det_dir = os.path.join(research_dir, 'object_detection')
for p in [research_dir, obj_det_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Check tensorflow version
from packaging import version
if version.parse(tf.__version__) < version.parse('1.4.0'):
    raise ImportError('Please update your TensorFlow installation to v1.4.* or later!')


ENV SETUP


In [5]:
# This is neaded to display the images.
%matplotlib inline 

Object detection imports

Here the imports for the object detection module


In [6]:
# import os, sys
# research_dir = os.path.abspath(os.path.join('..', 'models', 'research'))
# obj_det_dir = os.path.join(research_dir, 'object_detection')
# for p in [research_dir, obj_det_dir]:
#     if p not in sys.path:
#         sys.path.insert(0, p)

from utils import label_map_util
from utils import visualization_utils as vis_util


Model preparation

Variables 

for the list of other module -> detection model zoo

In [8]:
# What model to download.
MODEL_NAME = 'ssd_mobilenet_v1_coco_2017_11_17'
MODEL_FILE = MODEL_NAME + '.tar.gz'
DOWNLOAD_BASE = 'http://download.tensorflow.org/models/object_detection/'

# Path to frozen detection graph. This is the actual model that is used for the object detection.
PATH_TO_CKPT = MODEL_NAME + '/frozen_inference_graph.pb'

# List of the strings that is used to add correct label for each box.
PATH_TO_LABELS = os.path.join('data', 'mscoco_label_map.pbtxt')

NUM_CLASSES = 90


Download Module 

In [9]:
# Download model if not already present
if not os.path.exists(MODEL_FILE):
    urllib.request.urlretrieve(DOWNLOAD_BASE + MODEL_FILE, MODEL_FILE)

# Extract frozen inference graph
with tarfile.open(MODEL_FILE) as tar_file:
    for file in tar_file.getmembers():
        file_name = os.path.basename(file.name)
        if 'frozen_inference_graph.pb' in file_name:
            tar_file.extract(file, os.getcwd(), filter='data')


Load a (Frozen) Tensorflow model into memory

In [10]:
detection_graph = tf.Graph()
with detection_graph.as_default():
    od_graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(PATH_TO_CKPT, 'rb') as fid:
        serialized_graph = fid.read()
        od_graph_def.ParseFromString(serialized_graph)
        tf.compat.v1.import_graph_def(od_graph_def, name='')


Loading label map


In [11]:
label_map = label_map_util.load_labelmap(PATH_TO_LABELS)
categories = label_map_util.convert_label_map_to_categories(label_map, max_num_classes=NUM_CLASSES, use_display_name=True)
category_index = label_map_util.create_category_index(categories)

Helper code


In [12]:
def load_image_into_numpy_array(image):
    (im_width, im_height) = image.size
    return np.array(image.getdata()).reshape((im_height, im_width, 3)).astype(np.uint8)

Detection

In [13]:
from datasets.download_and_convert_cifar10 import _IMAGE_SIZE
# For the shake of simplisity we only use 2 images:
# image1.jpg
# image2.jpg
# If you want to test the code with your images, just add path to the images to the TEST_IMAGE_PATHS. 
PATH_TO_THE_IMAGES_DIR = 'test_images'
TEST_IMAGE_PATHS = [os.path.join(PATH_TO_THE_IMAGES_DIR, 'beach{}.jpg'.format(i)) for i in range(1,3) ]

# Size in inches, of the output images.
_IMAGE_SIZE = (12, 0)

In [20]:
from datetime import datetime

with detection_graph.as_default():
    with tf.compat.v1.Session(graph=detection_graph) as sess:
        # Definite input and output Tensors for detection_graph
        image_tensor = detection_graph.get_tensor_by_name('image_tensor:0')
        detection_boxes = detection_graph.get_tensor_by_name('detection_boxes:0')
        detection_scores = detection_graph.get_tensor_by_name('detection_scores:0')
        detection_classes = detection_graph.get_tensor_by_name('detection_classes:0')
        num_detections = detection_graph.get_tensor_by_name('num_detections:0')

        # Path to input video in Video folder and output annotated video
        input_video_path = os.path.join('Video', 'traffic.mp4')
        output_video_path = os.path.join('Video', 'traffic_annotated.mp4')

        video_reader = imageio.get_reader(input_video_path)
        fps = video_reader.get_meta_data().get('fps', 10)
        video_writer = imageio.get_writer(output_video_path, fps=fps)

        print(f"Processing video: {input_video_path}...")

        # Loop through and process each frame
        t0 = datetime.now()
        n_frames = 0
        for frame in video_reader:
            image_np = frame
            n_frames += 1

            # Expand dimensions since the model expects images to have shape: [1, None, None, 3]
            image_np_expanded = np.expand_dims(image_np, axis=0)

            # Actual detection
            (boxes, scores, classes, num) = sess.run(
                [detection_boxes, detection_scores, detection_classes, num_detections],
                feed_dict={image_tensor: image_np_expanded}
            )

            # Visualization of the results of a detection
            vis_util.visualize_boxes_and_labels_on_image_array(
                image_np,
                np.squeeze(boxes),
                np.squeeze(classes).astype(np.int32),
                np.squeeze(scores),
                category_index,
                use_normalized_coordinates=True,
                line_thickness=4
            )

            # Write annotated frame
            video_writer.append_data(image_np)

            if n_frames % 50 == 0:
                print(f"Processed {n_frames} frames...")

        processing_time = (datetime.now() - t0).total_seconds()
        fps_speed = n_frames / processing_time if processing_time > 0 else 0
        print(f"Completed! Frames processed: {n_frames}, Speed: {fps_speed:.2f} fps")
        print(f"Output saved to: {output_video_path}")

        # Cleanup
        video_reader.close()
        video_writer.close()


Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Completed! Frames processed: 400, Speed: 11.36 fps
Output saved to: Video\traffic_annotated.mp4
